# Actividad: XGBoost — de la intuición del boosting a un modelo real

**XGBoost** (*Extreme Gradient Boosting*) es uno de los algoritmos más usados en la práctica, durante años fue, por mucho, el más común entre los equipos ganadores de competencias de datos (Kaggle y similares), y sigue siendo un estándar en la industria para datos tabulares.

Esta actividad tiene dos partes:

1. **Construir la intuición a mano**, con un ejemplo de juguete muy pequeño, antes de usar ninguna librería especializada.
2. **Aplicar `XGBRegressor`** (la implementación real) a un dataset real, explorando los controles que más importan: cuántos árboles usar, qué tan "chicos" deben ser, cuándo parar, y cómo regularizar.

In [ ]:
# ============================================================
# PREPARACIÓN — no modifiquen esta celda
# ============================================================
# Si esta celda falla con "No module named 'xgboost'", corran antes, en una
# celda o terminal: pip install xgboost
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

print("XGBoost cargado correctamente.")

## Parte 0 — ¿Qué hace "boosting", en realidad?

La idea central de *boosting* es engañosamente simple:

1. Empiecen con una predicción muy simple (por ejemplo, el promedio de todos los datos).
2. Midan qué tan mal está esa predicción para cada dato -- eso es el **residual** (error = valor real − predicción).
3. Entrenen un árbol **muy simple** (poca profundidad) para predecir esos residuales, no los datos originales.
4. Sumen la predicción de ese árbol a la predicción acumulada -- pero multiplicada por un número chico (el **`learning_rate`**), para no pasarse.
5. Repitan: calculen los nuevos residuales (ya más chicos) y entrenen otro árbol simple para predecirlos.

Cada árbol individual es un **"weak learner"** (aprendiz débil) -- por sí solo, no sirve de mucho. Lo que hace poderoso al método es sumar cientos de ellos, cada uno corrigiendo un poquito el error que dejaron los anteriores. Eso es exactamente lo que hace XGBoost por dentro, con muchas optimizaciones extra. Vamos a hacerlo a mano, con un ejemplo de juguete, para verlo con sus propios ojos antes de usar la librería.

In [ ]:
# ============================================================
# Ejemplo de juguete — no modificar
# ============================================================
rng = np.random.RandomState(11)
x = np.linspace(0, 10, 50).reshape(-1, 1)

def funcion_real(xx):
    return (xx - 5)**2 / 3.0 + 1.5*np.sin(xx)

y = funcion_real(x.ravel()) + rng.normal(0, 1.2, size=50)

x_grid = np.linspace(0, 10, 300).reshape(-1, 1)

learning_rate = 0.5
n_rondas = 20

pred = np.full_like(y, y.mean())  # (300,1)
pred_grid = np.full(len(x_grid), y.mean()) #(300,)
snapshots = {}
suma_residuales_cuadrados = [np.sum((y - pred)**2)]

for ronda in range(1, n_rondas + 1):
    residual = y - pred                                  # paso 2: qué tan mal vamos
    arbolito = DecisionTreeRegressor(max_depth=2, random_state=0)
    arbolito.fit(x, residual)                             # paso 3: aprender el error
    pred = pred + learning_rate * arbolito.predict(x)               # paso 4: sumar con paso chico
    pred_grid = pred_grid + learning_rate * arbolito.predict(x_grid)
    suma_residuales_cuadrados.append(np.sum((y - pred)**2))
    if ronda in (1, 3, 8, 20):
        snapshots[ronda] = pred_grid.copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].scatter(x, y, color="black", s=25, label="datos", zorder=5)
axes[0].plot(x_grid, funcion_real(x_grid.ravel()), "--", color="gray", linewidth=1, label="función real (sin ruido)")
colores = plt.cm.viridis(np.linspace(0.15, 0.85, len(snapshots)))
for (ronda, pg), c in zip(snapshots.items(), colores):
    axes[0].plot(x_grid, pg, color=c, label=f"predicción tras ronda {ronda}")
axes[0].set_xlabel("x"); axes[0].set_ylabel("y"); axes[0].legend(fontsize=8)
axes[0].set_title("Predicción acumulada tras cada ronda de boosting")

axes[1].plot(range(len(suma_residuales_cuadrados)), suma_residuales_cuadrados, "o-", color="darkred")
axes[1].set_xlabel("ronda de boosting"); axes[1].set_ylabel("suma de residuales al cuadrado")
axes[1].set_title("El error total baja en cada ronda")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Suma de residuales² -- ronda 1: {suma_residuales_cuadrados[1]:.1f}   ronda 20: {suma_residuales_cuadrados[-1]:.1f}")

**Observación** Cada `arbolito` individual es de profundidad 2 -- prácticamente no sirve para nada por sí solo. Pero al sumar 20 de ellos, cada uno corrigiendo el error del anterior, la predicción acumulada se acerca bastante a la función real. Fíjense también que la predicción se ve "escalonada" -- son puros árboles pequeños sumados, no una curva suave.

Eso es exactamente la idea detrás de XGBoost: en vez de un árbol grande y complejo, muchísimos árboles chicos, cada uno corrigiendo un poco al anterior, sumados con un paso (`learning_rate`) controlado.

## Parte 1 — El dataset real: costos de seguro médico

Vamos a usar el dataset **Medical Cost Personal Datasets** (a veces llamado simplemente *insurance*), muy usado para practicar regresión con variables mixtas (numéricas y categóricas). Tiene 1,338 personas, con estas columnas:

| Columna | Significado |
|---|---|
| `age` | edad |
| `sex` | sexo |
| `bmi` | índice de masa corporal |
| `children` | número de hijos/dependientes |
| `smoker` | si la persona fuma o no |
| `region` | región de EE.UU. donde vive |
| `charges` | **costo anual del seguro médico (lo que queremos predecir)** |

Es un dataset chico y fácil de entender -- todos tenemos una intuición de qué variables deberían importar (¿fumar debería subir el costo? ¿la edad?), lo cual lo hace bueno para revisar si el modelo "tiene sentido".

In [ ]:
# ============================================================
# PREPARACIÓN — no modifiquen esta celda
# ============================================================
url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
df = pd.read_csv(url)

print(df.shape)
print(df.isna().sum().sum(), "valores faltantes")
df.head()

In [ ]:
# ============================================================
# PREPARACIÓN — no modifiquen esta celda
# ============================================================
# sex, smoker y region son categóricas -- las convertimos a columnas 0/1
# Nota: codificamos antes de partir el conjunto de datos a propósito; 
# si partieran primero, se usaría OneHotEncoder con fit solo en train
df_cod = pd.get_dummies(df, columns=["sex", "smoker", "region"], drop_first=True)

X = df_cod.drop(columns=["charges"])
y = df_cod["charges"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Columnas de X:", list(X.columns))
print("Entrenamiento:", X_train.shape, "  Prueba:", X_test.shape)

## Parte 2 — Línea base: regresión lineal y un árbol sin restricciones

Antes de usar XGBoost, midan contra qué lo van a comparar. Ya conocen ambos modelos de actividades anteriores del curso -- corran esta celda y anoten los números.

In [ ]:
modelo_lineal = LinearRegression().fit(X_train, y_train)
mse_lineal = mean_squared_error(y_test, modelo_lineal.predict(X_test))
r2_lineal = r2_score(y_test, modelo_lineal.predict(X_test))
print(f"Regresión lineal       -- MSE prueba: {mse_lineal:>12,.0f}   R2: {r2_lineal:.4f}")

arbol_default = DecisionTreeRegressor(random_state=42).fit(X_train, y_train)
mse_arbol_train = mean_squared_error(y_train, arbol_default.predict(X_train))
mse_arbol = mean_squared_error(y_test, arbol_default.predict(X_test))
r2_arbol_train = r2_score(y_train, arbol_default.predict(X_train))
r2_arbol = r2_score(y_test, arbol_default.predict(X_test))
print(f"Árbol sin restricciones -- MSE entrenamiento: {mse_arbol_train:>10,.0f}  (profundidad {arbol_default.get_depth()}, {arbol_default.get_n_leaves()} hojas)")
print(f"Árbol sin restricciones -- R2 entrenamiento: {r2_arbol_train:.4f}")
print(f"Árbol sin restricciones -- MSE prueba: {mse_arbol:>15,.0f}")
print(f"Árbol sin restricciones -- R2 prueba: {r2_arbol:.4f}")


**Fíjense bien:** el árbol sin restricciones memoriza el entrenamiento casi perfecto, pero en prueba queda **peor** que la regresión lineal sin ningún ajuste. Ya vimos este patrón antes con árboles -- un modelo muy flexible sin control no necesariamente generaliza mejor. Veamos si XGBoost, que también usa árboles, logra hacerlo mejor.

## Parte 3 — XGBoost con parámetros por default

**Su turno.** Creen un `XGBRegressor(random_state=42)`, entrénenlo con `X_train, y_train`, y calculen su MSE y R2 de prueba, igual que arriba.

In [ ]:
# TODO: creen y entrenen un XGBRegressor(random_state=42) llamado xgb_default,
# y calculen su MSE y R2 de prueba (igual que con los modelos anteriores)

# xgb_default = ...
# xgb_default.fit(...)
# mse_xgb_default = ...
# r2_xgb_default = ...
# print(...)


In [ ]:
XGBRegressor(random_state=42)

**Para anotar:** ¿le ganó XGBoost a la regresión lineal? ¿Y al árbol sin restricciones? Con parámetros completamente por default (sin que ustedes tocaran nada), XGBoost ya debería estar considerablemente mejor que los dos.

## Parte 4 — Cómo aprende boosting: la curva de error por ronda

Igual que en el ejemplo de juguete, XGBoost va sumando árboles uno por uno. Podemos pedirle que **registre el error en cada ronda**, tanto en entrenamiento como en un conjunto de evaluación, usando `eval_set`. Esta celda usa un `learning_rate` alto (0.3) y árboles no tan chicos (`max_depth=6`) a propósito, para que el sobreajuste se note.

In [ ]:
# ============================================================
# Ya lista — no modificar
# ============================================================
modelo_curva = XGBRegressor(
    n_estimators=300, learning_rate=0.3, max_depth=6,
    random_state=42, eval_metric="rmse"
)
modelo_curva.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False
)
resultados_curva = modelo_curva.evals_result()
rmse_train = resultados_curva["validation_0"]["rmse"]
rmse_test = resultados_curva["validation_1"]["rmse"]

plt.figure(figsize=(8, 5))
plt.plot(rmse_train, label="entrenamiento")
plt.plot(rmse_test, label="prueba")
mejor_ronda = int(np.argmin(rmse_test))
plt.axvline(mejor_ronda, color="gray", linestyle="--", linewidth=1,
            label=f"mejor ronda en prueba ({mejor_ronda})")
plt.xlabel("ronda de boosting (árbol #)")
plt.ylabel("RMSE")
plt.title("learning_rate=0.3, max_depth=6 -- boosting sin freno")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Mejor RMSE de prueba: {min(rmse_test):.1f}, en la ronda {mejor_ronda}")
print(f"RMSE de prueba en la última ronda (300): {rmse_test[-1]:.1f}")

**¿Qué observan?** El error de entrenamiento sigue bajando toda la gráfica (más árboles = ajusta mejor el entrenamiento, siempre). Pero el error de **prueba** llega a su mínimo muy pronto y después empieza a subir otra vez -- después de ese punto, cada árbol nuevo solo está ayudando a memorizar ruido del entrenamiento. Es el mismo patrón de sobreajuste de siempre, solo que ahora ocurre "ronda por ronda" en vez de "profundidad por profundidad".

## Parte 5 — Los controles clave: `learning_rate`, `max_depth`, y cuándo parar

Tres formas de evitar lo que acaban de ver:

- **`learning_rate` chico** (ej. 0.05 en vez de 0.3): cada árbol corrige un poquito menos, así que hacen falta más rondas para llegar al mismo lugar, pero el camino es más suave y es más difícil pasarse.
- **`max_depth` chico** (ej. 2-4): en boosting, los árboles individuales deben ser simples a propósito -- son "weak learners". Nada que ver con dejarlos crecer libres como en un árbol solo.
- **`early_stopping_rounds`**: en vez de adivinar cuántas rondas usar, le dicen a XGBoost "paren si el error de validación no mejora en N rondas seguidas" -- así el propio algoritmo encuentra el punto óptimo que ustedes vieron marcado en la gráfica anterior.

**Su turno:** usando el mismo patrón del `eval_set`, entrenen un `XGBRegressor` con `n_estimators=1000`, `learning_rate=0.05`, `max_depth=3`, `eval_metric="rmse"` y `early_stopping_rounds=20`. **Importante:** para no hacer trampa, no usen `X_test` para decidir cuándo parar -- separen un conjunto de *validación* a partir del propio `X_train`:

```python
X_train2, X_val, y_train2, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42)
```

Entrenen con `eval_set=[(X_val, y_val)]`, y al final evalúen el modelo (ya con su `best_iteration` elegida automáticamente) contra el `X_test` original, que nunca tocaron.

In [ ]:
# TODO: separen un set de validación de X_train, entrenen con early stopping,
# y evalúen el modelo final (llámenlo xgb_early) contra X_test

# X_train2, X_val, y_train2, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42)
# xgb_early = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=3,
#                           random_state=42, eval_metric="rmse", early_stopping_rounds=20)
# xgb_early.fit(X_train2, y_train2, eval_set=[(X_val, y_val)], verbose=False)
# print("Mejor ronda encontrada:", xgb_early.best_iteration)
# mse_xgb_early = ...
# r2_xgb_early = ...


## Parte 6 — Regularización L1 / L2 (`reg_alpha`, `reg_lambda`)

Además de controlar cuántos árboles usar y qué tan profundos, XGBoost puede penalizar directamente los valores que caen en cada hoja del árbol -- la misma idea de "penalizar coeficientes grandes" que existe en regresión (Ridge = L2, Lasso = L1), aplicada aquí a las hojas en vez de a coeficientes:

- **`reg_lambda`** (L2): encoge los valores de las hojas hacia cero, suavemente.
- **`reg_alpha`** (L1): puede llevar algunas hojas a exactamente cero.

Esta celda usa un modelo con más capacidad de la que necesita (`max_depth=6`, `n_estimators=300`, sin `learning_rate` chico) a propósito, para que se note el efecto de regularizar.

In [ ]:
# ============================================================
# Ya lista — no modificar
# ============================================================
valores_lambda = [0, 1, 5, 20, 50, 100]
mse_por_lambda = []

for rl in valores_lambda:
    m = XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6,
                      random_state=42, reg_lambda=rl)
    m.fit(X_train, y_train)
    mse_por_lambda.append(mean_squared_error(y_test, m.predict(X_test)))

plt.figure(figsize=(7, 4.5))
plt.plot(valores_lambda, mse_por_lambda, "o-")
plt.xlabel("reg_lambda")
plt.ylabel("MSE de prueba")
plt.title("Efecto de la regularización L2 (modelo con mucha capacidad)")
plt.grid(alpha=0.3)
plt.show()

for rl, mse in zip(valores_lambda, mse_por_lambda):
    print(f"reg_lambda={rl:>4}: MSE prueba = {mse:,.0f}")

**¿Qué observan?** Con un modelo que ya tiene bastante capacidad (árboles de profundidad 6, 300 de ellos), subir `reg_lambda` mejora el MSE de prueba de forma consistente -- la regularización está ayudando a que el modelo no sobreajuste tanto. (Si prueban esto con un modelo que ya era chico y conservador, probablemente no van a ver ninguna mejora -- no hay mucho que regularizar si el modelo ya no se estaba sobreajustando.)

## Parte 7 — Búsqueda de hiperparámetros con `RandomizedSearchCV`

Ya vieron varios controles por separado (`learning_rate`, `max_depth`, `reg_lambda`, `reg_alpha`...). En la práctica no se ajustan uno por uno -- se buscan varias combinaciones a la vez con validación cruzada. Con tantos parámetros posibles, probar *todas* las combinaciones (`GridSearchCV`) sería carísimo -- por eso aquí usamos `RandomizedSearchCV`, que prueba una muestra aleatoria de combinaciones.

**Su turno:** usando el `param_dist` ya definido abajo, construyan un `RandomizedSearchCV` sobre `XGBRegressor(random_state=42)`, con `n_iter=40`, `cv=5`, `scoring="neg_mean_squared_error"` y `random_state=42`. Ajusten (`fit`) sobre `X_train, y_train`.

In [ ]:
# ============================================================
# Ya lista — no modificar
# ============================================================
param_dist = {
    "n_estimators": [100, 200, 300, 500, 800],
    "max_depth": [2, 3, 4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_lambda": [0, 1, 5, 20, 50],
    "reg_alpha": [0, 1, 5, 20],
}

In [ ]:
# TODO: construyan y ajusten el RandomizedSearchCV (llámenlo busqueda),
# y guarden el mejor modelo en xgb_tuned

# busqueda = RandomizedSearchCV(
#     XGBRegressor(random_state=42), param_distributions=param_dist,
#     n_iter=40, cv=5, scoring="neg_mean_squared_error", random_state=42, n_jobs=-1
# )
# busqueda.fit(X_train, y_train)
# print("Mejores parámetros:", busqueda.best_params_)
# xgb_tuned = busqueda.best_estimator_


## Parte 8 — Importancia de variables

Una ventaja de los modelos basados en árboles (incluyendo XGBoost) es que pueden reportar qué tanto usó cada variable para hacer sus divisiones -- es una forma rápida de interpretabilidad, aunque el modelo en sí sea una suma de cientos de árboles.

In [ ]:
importancias = pd.Series(xgb_tuned.feature_importances_, index=X.columns).sort_values()

plt.figure(figsize=(7, 5))
importancias.plot(kind="barh", color="teal")
plt.xlabel("importancia")
plt.title("Importancia de variables -- XGBoost afinado")
plt.tight_layout()
plt.show()

print(importancias.sort_values(ascending=False))

## Parte 9 — Cierre: comparando todos los modelos

Reúnan aquí, contra el conjunto de **prueba**, todos los modelos que construyeron en esta actividad.

In [ ]:
resultados = []
for nombre, modelo in [
    ("Regresión lineal", modelo_lineal),
    ("Árbol sin restricciones", arbol_default),
    ("XGBoost (default)", xgb_default),
    ("XGBoost (early stopping)", xgb_early),
    ("XGBoost (tuneado)", xgb_tuned),
]:
    pred = modelo.predict(X_test)
    mse = mean_squared_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    resultados.append((nombre, mse, r2))

print(f"{'Modelo':30s} {'MSE prueba':>14s} {'R2 prueba':>12s}")
for nombre, mse, r2 in resultados:
    print(f"{nombre:30s} {mse:14,.0f} {r2:12.4f}")

## Para discutir

1. En la Parte 2, el árbol sin restricciones quedó **peor** que la regresión lineal en prueba, a pesar de memorizar perfecto el entrenamiento. ¿Por qué XGBoost, que también usa árboles por dentro, no tiene el mismo problema con sus parámetros por default?

2. En la gráfica de la Parte 4, ¿en qué ronda empieza a subir el error de prueba? ¿Qué le pasa al error de entrenamiento en ese mismo punto?

3. Comparen el modelo con `early_stopping_rounds` contra el modelo afinado con `RandomizedSearchCV`. ¿Cuál dio mejor resultado? Si `RandomizedSearchCV` prueba muchas combinaciones con validación cruzada, ¿por qué no necesariamente le gana a un solo modelo bien elegido con early stopping? (Pista: piensen en cuántas combinaciones se probaron realmente, y qué tan grande es el dataset.)

4. Miren la gráfica de importancia de variables. ¿Tiene sentido que esa variable sea la más importante para predecir el costo de un seguro médico? ¿Qué otras variables del mundo real (que no están en este dataset) esperarían que también importen?

5. `max_depth` significa cosas distintas en un árbol de decisión solo (donde profundidad = capacidad de memorizar) y en XGBoost (donde cada árbol es deliberadamente poco profundo). En sus propias palabras, ¿por qué tiene sentido esta diferencia?

6. Si tuvieran que explicarle a alguien que no sabe nada de Machine Learning qué hace "boosting", usando el ejemplo de juguete de la Parte 0, ¿cómo se lo explicarían en dos o tres frases?